# VWAP时段选择预测 - 日内特征研究框架

## 核心思路：利用集合竞价和开盘5分钟数据预测当日VWAP差异

---

### 核心假设
- **集合竞价**反映了隔夜信息的消化和市场情绪
- **开盘5分钟**的量价特征能预示当日的日内走势模式
- 开盘阶段的异常信号（如放量、跳空）对后续走势有预测力

---

## 一、问题定义与Label选择

### 1.1 目标变量定义

$$Y_T = \frac{VWAP_{945-1000am,T} - VWAP_{1445-1500pm,T}}{Close_{T-1}}$$




---

## 二、日内特征因子清单（附公式与逻辑）

以下因子均可用5分钟数据计算，分为三类：

---

### 1. 集合竞价阶段因子（9:15-9:25）

| 因子名称 | 公式/计算方法 | 经济逻辑 | 预测机制 |
|----------|--------------|----------|----------|
| **隔夜订单不平衡率** | $(竞价买单量 - 竞价卖单量) / (竞价买单量 + 竞价卖单量)$ | 反映隔夜多空力量对比 | >0预示买方力量强，开盘可能跳涨，上午VWAP可能高于下午 |
| **集合竞价成交量比** | $竞价成交量 / 前日日均每分钟成交量$ | 衡量开盘参与热度 | 异常放量暗示方向选择强度，高成交量比往往伴随趋势延续 |
| **价格发现效率** | $开盘价 - 前日收盘价$ | 衡量隔夜信息冲击程度 | 大幅跳空后日内往往有回归，影响上下午VWAP差异 |
| **竞价振幅** | $(竞价最高价 - 竞价最低价) / 前日收盘价$ | 衡量开盘前的价格波动 | 高振幅意味着多空分歧大，日内走势可能更剧烈 |
| **开盘相对位置** | $(开盘价 - 竞价最低价) / (竞价最高价 - 竞价最低价)$ | 开盘价在竞价区间的位置 | 接近高点开盘暗示买方占优，可能利好上午VWAP |

---

### 2. T开盘,T-1收盘15分钟因子（9:30-9:45,14:30-14:45）

| 因子名称 | 公式/计算方法 | 经济逻辑 | 预测机制 |
|----------|--------------|----------|----------|
| **开盘跳涨幅度** | $(开盘价 - 前日收盘价) / 前日收盘价$ | 方向性情绪指标 | 跳空方向往往与日内趋势相关，但也可能触发回补 |
| **首5分钟成交量冲击** | $首5分钟成交量 / 前5日同期均值$ | 异常资金流入流出信号 | 放量突破往往有延续性，缩量则信号弱 |
| **价格回归强度** | $max(首5分钟高点, 开盘价) - min(首5分钟低点, 开盘价)$ | 衡量多空博弈激烈程度 | 高振幅意味着分歧大，可能预示日内反转 |
| **VWAP斜率** | $(第5分钟VWAP - 开盘价) / 开盘价$ | 趋势持续性早期信号 | 正斜率暗示买盘持续流入，利好上午VWAP |
| **首5分钟收盘位置** | $(首5分钟收盘 - 首5分钟最低) / (首5分钟最高 - 首5分钟最低)$ | K线形态强弱 | 收在高位暗示多头占优，收在低位暗示空头占优 |
| **跳空回补程度** | 跳空后5分钟内回补的比例 | 衡量跳空的持续性 | 快速回补暗示跳空无效，日内可能反转 |

---

### 3. 开盘动量与趋势因子

| 因子名称 | 公式/计算方法 | 经济逻辑 | 预测机制 |
|----------|--------------|----------|----------|
| **开盘动量** | $(首5分钟收盘 - 前日收盘) / 前日收盘$ | 综合跳空和首5分钟走势 | 强动量往往延续，但极端动量可能反转 |
| **趋势延续信号** | $sign(开盘涨跌幅) × sign(首5分钟涨跌幅)$ | 判断开盘后是否延续 | +1表示延续，-1表示开盘后反转 |
| **成交额集中度** | $首5分钟成交额 / 日均成交额$ | 开盘交易活跃度 | 高集中度意味着开盘定价重要性高 |

---

### 4. 历史模式因子

| 因子名称 | 公式/计算方法 | 经济逻辑 | 预测机制 |
|----------|--------------|----------|----------|
| **Spread历史均值** | $过去N日Spread的均值$ | 个股的固有模式 | 某些股票长期倾向于上午或下午更活跃 |
| **Spread波动率** | $过去N日Spread的标准差$ | 预测难度指标 | 高波动意味着预测困难，需要更保守的策略 |
| **Spread自相关** | $Spread_{t-1}与Spread_t的相关性$ | 动量/反转特征 | 正相关暗示动量，负相关暗示均值回归 |

---


### 因子预期作用总结

| 类别 | 核心因子 | 对Spread的预期影响 |
|------|----------|-------------------|
| 集合竞价 | 订单不平衡率、价格发现效率 | 买方力量强 → 上午VWAP高 → Spread正 |
| 开盘5分钟 | VWAP斜率、成交量冲击 | 正斜率+放量 → 趋势延续 → 影响Spread方向 |
| 动量 | 开盘动量、趋势延续信号 | 强动量延续 → Spread与动量方向一致 |
| 历史 | Spread均值 | 回归历史均值 |
| 时间 | 周内效应 | 周一波动大，Spread可能更极端 |
"""

---

## 三、模型训练与验证

---

## 三、模型选择建议

### 3.1 阶段一：基线模型 (Baseline)
- 简单历史均值策略
- 假设spread_20d_mean是一个特征
- predicted_spread = spread_20d_mean

**目的**：验证日内模式是否有延续性

---

### 3.3 阶段二：树模型 (LightGBM)

**进阶选择**

优点：
- 自动捕捉非线性关系
- 特征重要性分析
- 处理缺失值

适用场景：特征数量 > 20，数据量充足

---

## 四、评估指标

| 指标 | 公式 | 说明 |
|------|------|------|
| IC (Information Coefficient) | $Corr(Y_{pred}, Y_{actual})$ | 预测能力，>0.05有意义 |
| IC_IR | $Mean(IC) / Std(IC)$ | IC稳定性，>0.5较好 |
| Hit Rate | 预测方向正确的比例 | >55%有统计显著性 |
| IS (Implementation Shortfall) | $(实际成交均价 / VWAP全天) - 1$ | <0表示策略优于VWAP |
| 年化节约成本 | $Mean(IS) \times 252 \times 年交易量$ | 经济价值 |